# Taller 1: Sistema masa-resorte-amortiguador de 2 GDL con excitacion de base

Este notebook describe y reproduce el proceso del README en la carpeta Taller 1.

## Objetivo general
Modelar y simular la dinamica vertical de un sistema mecanico de 2 GDL con entrada de base senoidal y analizar la respuesta de x1(t) y x2(t).

Voy a modificar ahora mismo la celda de ecuaciones en la notebook para dejarla en formato LaTeX como indicaste, y enseguida te confirmo el cambio exacto y la ruta.

## 1. Descripcion del sistema

- m1: masa no suspendida (rueda/eje).
- m2: masa suspendida (chasis).
- k1: rigidez del neumatico respecto a la base.
- k2: rigidez de la suspension entre m1 y m2.
- c: amortiguamiento viscoso entre m1 y m2.
- y(t): excitacion de base, tipicamente y(t) = A sin(omega t).

## 2. Ecuaciones diferenciales

Ecuaciones del sistema:

$$
m_1 \ddot{x}_1 + c(\dot{x}_1-\dot{x}_2) + k_2(x_1-x_2) + k_1(x_1-y)=0
$$

$$
m_2 \ddot{x}_2 + c(\dot{x}_2-\dot{x}_1) + k_2(x_2-x_1)=0
$$

## 3. Forma matricial

M*x_ddot + C*x_dot + K*x = f(t)

M = [[m1, 0], [0, m2]]
C = [[c, -c], [-c, c]]
K = [[k1+k2, -k2], [-k2, k2]]
f(t) = [k1*y(t), 0]^T

## 4. Condiciones iniciales

Para simulacion desde reposo:

x1(0)=0, x1_dot(0)=0, x2(0)=0, x2_dot(0)=0

In [ ]:
# Importaciones
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
# 5. Definicion de parametros (ajustables)
m1 = 40.0      # kg
m2 = 300.0     # kg
k1 = 180000.0  # N/m
k2 = 18000.0   # N/m
c = 1200.0     # N*s/m

A = 0.02       # m (amplitud de base)
omega = 2*np.pi*2.0  # rad/s (2 Hz)

t0, tf = 0.0, 10.0
t_eval = np.linspace(t0, tf, 3000)

In [ ]:
# 6. Modelo en espacio de estado
# Estado: z = [x1, x1_dot, x2, x2_dot]
def y_base(t):
    return A * np.sin(omega * t)

def modelo(t, z):
    x1, x1_dot, x2, x2_dot = z
    y = y_base(t)

    x1_ddot = (-c*(x1_dot - x2_dot) - k2*(x1 - x2) - k1*(x1 - y)) / m1
    x2_ddot = (-c*(x2_dot - x1_dot) - k2*(x2 - x1)) / m2

    return [x1_dot, x1_ddot, x2_dot, x2_ddot]

In [ ]:
# 7. Simulacion
z0 = [0.0, 0.0, 0.0, 0.0]
sol = solve_ivp(modelo, (t0, tf), z0, t_eval=t_eval, method='RK45')

t = sol.t
x1 = sol.y[0]
x2 = sol.y[2]
y = y_base(t)

In [ ]:
# 8. Graficas principales
plt.figure(figsize=(10, 6))
plt.plot(t, y, label='y(t): base', linewidth=1.5)
plt.plot(t, x1, label='x1(t): masa no suspendida', linewidth=1.2)
plt.plot(t, x2, label='x2(t): masa suspendida', linewidth=1.2)
plt.title('Respuesta temporal del sistema 2 GDL')
plt.xlabel('Tiempo [s]')
plt.ylabel('Desplazamiento [m]')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 9. Metricas simples de desempeno
amp_x1 = np.max(np.abs(x1))
amp_x2 = np.max(np.abs(x2))
transmision = amp_x2 / A if A != 0 else np.nan

print(f'Amplitud maxima |x1|: {amp_x1:.6f} m')
print(f'Amplitud maxima |x2|: {amp_x2:.6f} m')
print(f'Indice simple de transmision |x2|/A: {transmision:.3f}')

## 10. Relacion con los objetivos del README

1. Se plantea el modelo dinamico continuo de 2 GDL.
2. Se implementan las ecuaciones diferenciales acopladas.
3. Se simula la respuesta temporal para entrada senoidal de base.
4. Se habilita el analisis de sensibilidad variando m1, m2, k1, k2 y c.
5. Se observan amplitudes y respuesta de x2(t) como indicador de confort.